# 05 — Model Training

Trains Logistic Regression, Random Forest, and XGBoost on the feature-engineered loan data and compares
them.

**Leakage note:** `drop_leaky_columns` now runs between cleaning and feature engineering, removing
post-origination/hardship/servicing-time columns (`credit_risk.data.leakage.LEAKY_COLUMNS`) before the
model ever sees them. `prepare_model_matrix`'s own drop list still runs afterward for the non-leakage
columns it handles (the target-source `loan_status`, identifiers, raw date columns already superseded by
`credit_history_years`) — most of its leakage-column entries are now redundant no-ops since
`drop_leaky_columns` already removed them upstream, which is fine since it's a no-op, not a conflict.

This is a real fix, not a no-op: `funded_amnt`, `funded_amnt_inv`, and the entire `hardship_*` family were
previously reaching the model unflagged.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_curve
from sklearn.model_selection import train_test_split

from credit_risk.config import settings
from credit_risk.data import clean_and_label, drop_leaky_columns, load_raw_data
from credit_risk.features import build_features
from credit_risk.models import assign_risk_band, evaluate, prepare_model_matrix, train

## Build the model matrix

In [ ]:
df = load_raw_data(settings.raw_data_path)
df = clean_and_label(df)
df = drop_leaky_columns(df)
df = build_features(df)

X, y = prepare_model_matrix(df, target_column=settings.target_column)
print(X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=settings.random_seed, stratify=y
)
print(X_train.shape, X_test.shape)

## Baseline

In [ ]:
baseline_preds = np.zeros_like(y_test)
print("Baseline accuracy (always predict no default):", accuracy_score(y_test, baseline_preds))

## Train and evaluate all three models

`train()` dispatches on `model_name`; Logistic Regression is returned as a `StandardScaler +
LogisticRegression` pipeline so `evaluate()` can call `.predict_proba` on raw `X_test` for all three model
types without special-casing.

In [ ]:
models = {}
metrics = {}

for model_name in ["logistic", "random_forest", "xgboost"]:
    models[model_name] = train(X_train, y_train, model_name)
    metrics[model_name] = evaluate(models[model_name], X_test, y_test)

pd.DataFrame(metrics).T

## ROC curve comparison

In [ ]:
plt.figure(figsize=(8, 6))
for model_name, model in models.items():
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=model_name)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

## Feature importance (Random Forest)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": models["random_forest"].feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x="importance", y="feature")
plt.title("Top 15 Important Features")
plt.show()

## Risk bands (XGBoost)

In [ ]:
xgb_probs = models["xgboost"].predict_proba(X_test)[:, 1]
risk_scores = pd.DataFrame({"actual": y_test, "probability_default": xgb_probs})
risk_scores["risk_band"] = assign_risk_band(risk_scores["probability_default"])

risk_scores.groupby("risk_band", observed=True)["actual"].mean() * 100

In [ ]:
predictions_dir = settings.outputs_dir / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

risk_scores.to_csv(predictions_dir / "risk_scores.csv", index=False)
feature_importance.to_csv(predictions_dir / "feature_importance.csv", index=False)

## Business conclusion

XGBoost gave the strongest discriminatory performance of the three; Logistic Regression and Random Forest
landed close behind it. The most influential predictors were loan grade, sub-grade, interest rate, and loan
term, followed by the engineered repayment-capacity ratios (`income_to_loan_ratio`,
`installment_to_income`) — evidence that borrower leverage and LendingClub's own internal grading carry most
of the predictive signal, with the engineered features adding a real but secondary contribution.

The resulting risk bands separate cleanly from ~11% default in the lowest band to over 90% in the highest,
which is a strong basis for the threshold and decision-framework work in the next notebook.

## Save models

In [ ]:
import joblib
from datetime import datetime, timezone

models_dir = settings.outputs_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

for model_name, model in models.items():
    joblib.dump(model, models_dir / f"{model_name}_{timestamp}.pkl")

print("Saved 3 models to", models_dir)